# Huberman Lab Podcast

### Setup

In [1]:
import os
import pandas as pd
import pickle
from youtube_transcript_api import YouTubeTranscriptApi

In [2]:
ytt_api = YouTubeTranscriptApi()

In [3]:
transcript = ytt_api.fetch('4b6bwcWK6GE', ['en-US'])

In [4]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='- Welcome to the Huberman Lab Podcast', start=0.37, duration=1.94), FetchedTranscriptSnippet(text='where we discuss science', start=2.31, duration=1.39), FetchedTranscriptSnippet(text='and science-based tools for everyday life.', start=3.7, duration=2.243), FetchedTranscriptSnippet(text="Welcome, I'm Andrew Huberman,", start=9.49, duration=1.97), FetchedTranscriptSnippet(text="and I'm a professor of\nneurobiology and ophthalmology", start=11.46, duration=2.92), FetchedTranscriptSnippet(text='at Stanford School of Medicine.', start=14.38, duration=2.3), FetchedTranscriptSnippet(text="That's what I do for my day job,", start=16.68, duration=1.73), FetchedTranscriptSnippet(text='but over the last few years', start=18.41, duration=1.49), FetchedTranscriptSnippet(text="I've become very active\nin teaching neuroscience", start=19.9, duration=2.91), FetchedTranscriptSnippet(text='and neuroscience-related\nthemes on social media.', sta

In [5]:
data = pd.read_csv('data/huberman_videos.csv', encoding='latin1', index_col=False)
data.head()

,id,url,title,video_key
0,0,https://www.youtube.com/watch?v=4b6bwcWK6GE&li...,Welcome to the Huberman Lab Podcast,4b6bwcWK6GE
1,1,https://www.youtube.com/watch?v=H-XfCl-HpRM&li...,How Your Brain Works & Changes,H-XfCl-HpRM
2,2,https://www.youtube.com/watch?v=nm1TxQj9IsQ&li...,Master Your Sleep & Be More Alert When Awake,nm1TxQj9IsQ
3,3,https://www.youtube.com/watch?v=nwSkFq4tyC0&li...,"Using Science to Optimize Sleep, Learning & Me...",nwSkFq4tyC0
4,4,https://www.youtube.com/watch?v=NAATB55oxeQ&li...,"How to Defeat Jet Lag, Shift Work & Sleeplessness",NAATB55oxeQ


### Prepare a utility dict

This will be beneficial when creating the embeddings and allow us to pass the title of each video_id into the embedding module 

In [6]:
title_dict = data.set_index('video_key')['title'].to_dict()
print(f'Video title at index zero:\n{title_dict[data.video_key[:2][0]]}\n')
print(f'Video title at index one:\n{title_dict[data.video_key[:2][1]]}')

Video title at index zero:
Welcome to the Huberman Lab Podcast

Video title at index one:
How Your Brain Works & Changes


In [7]:
with open('data/title_dict.pkl', 'wb') as f:
    pickle.dump(title_dict, f)

### Generate and Store Transcripts

TODO: Update this to utilize `Mongo` opposed to it's current `.txt` storage

In [9]:
os.makedirs('data/documents', exist_ok=True)

for video_key in data['video_key']:
    filename = f'data/documents/{video_key}.txt'

    try:
        fetched_transcript = ytt_api.get_transcript(video_key, languages=['en', 'en-US'])
        
        with open(filename, 'a', encoding='utf-8') as file:
            for snippet in fetched_transcript:
                file.write(snippet['text'] + '\n')  # Add newline after each snippet
                
    except Exception as e:
        print(f"Error processing video {video_key}")